# Quill — Iteration 2
## Notebook 01: Cord Notation & Encoding

**The idea:** learn quantum computing through text.

### Our vocabulary
- **Knot:** a group of qubits treated as one unit. In this first version, one knot encodes one 8-bit ASCII character.
- **Cord:** an ordered sequence of knots, representing a string.
- **Cord notation:** a compact way to show knots and operations on them.

A knot is **not automatically entangled**. It is a grouping. Entanglement is a property of the quantum state, not a name or drawing.

**Scope:** notation and encoding only. Later notebooks will add transformations, superposition, rotations, and entanglement.

## 1. A qubit, in one equation

A qubit has computational-basis states $|0\rangle$ and $|1\rangle$. A general pure state can be written

$$
|\psi\rangle = \alpha |0\rangle + \beta |1\rangle,
\qquad |\alpha|^2 + |\beta|^2 = 1.
$$

The complex numbers $\alpha$ and $\beta$ are **amplitudes**; their squared magnitudes give measurement probabilities. We'll study superposition in a later notebook.

## 2. Set up Qiskit

In [ ]:
%pip install -q qiskit qiskit-aer
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
simulator = AerSimulator()
print("Quill is ready.")

## 3. Encode one character: a knot

For this version, a **knot contains 8 qubits** and encodes one ASCII character. `H` has decimal ASCII value 72:

$$
72_{10} = 01001000_2.
$$

We prepare this basis state by applying an $X$ gate wherever the bit is 1. The gate is

$$
X = \begin{pmatrix}0 & 1\\1 & 0\end{pmatrix},
\qquad X|0\rangle = |1\rangle,
\qquad X|1\rangle = |0\rangle.
$$

This is an encoding, not a quantum speedup: it maps ordinary text to a circuit state.

In [ ]:
BITS_PER_CHAR = 8

def encode_character(qc, char, start_qubit=0):
    """Prepare one ASCII character in 8 qubits, LSB-first by qubit index."""
    if len(char) != 1 or ord(char) > 127:
        raise ValueError("Use one ASCII character (code point 0–127).")
    value = ord(char)
    for bit_index in range(BITS_PER_CHAR):
        if (value >> bit_index) & 1:
            qc.x(start_qubit + bit_index)

knot_circuit = QuantumCircuit(8)
encode_character(knot_circuit, "H")
print("Character: H")
print("ASCII decimal:", ord("H"))
print("Binary, MSB first:", format(ord("H"), "08b"))
print(knot_circuit.draw())

### Knot notation — v0.1

```text
Knot H
┌─────────────────┐
│ q0 q1 ... q7    │  8 qubits encode one character
└─────────────────┘
```

The binary string is displayed most-significant bit first, while qubit index 0 is the least-significant bit. The code handles this convention.

## 4. Encode a string: a cord

A cord is an **ordered sequence of knots**. For `HI`, each character gets its own group of 8 qubits:

$$
\boxed{\text{Knot H}}\; - \;\boxed{\text{Knot I}}
\quad\Longrightarrow\quad 16\text{ qubits}.
$$

The dash means “next knot in the sequence”; it does **not** mean entanglement.

In [ ]:
def encode_cord(text):
    """Create one circuit with 8 qubits per ASCII character."""
    if any(ord(char) > 127 for char in text):
        raise ValueError("This first version supports ASCII only.")
    qc = QuantumCircuit(len(text) * BITS_PER_CHAR)
    for char_index, char in enumerate(text):
        encode_character(qc, char, start_qubit=char_index * BITS_PER_CHAR)
    return qc

cord_text = "HI"
cord_circuit = encode_cord(cord_text)
print("Cord:", cord_text)
print("Knots:", len(cord_text))
print("Qubits:", cord_circuit.num_qubits)
print(cord_circuit.draw())

## 5. What state did we prepare?

The circuit prepares a computational-basis state. We can label the state of the 16-qubit cord compactly as

$$
|HI\rangle_{\mathrm{cord}}.
$$

This is shorthand for one specific 16-qubit basis state prepared by the circuit. “Cord” is our teaching abstraction, not a new kind of physical state.

## 6. Measure and decode

In [ ]:
def decode_counts(counts, n_chars):
    decoded = {}
    for bitstring, count in counts.items():
        bits = bitstring.replace(" ", "")[::-1]  # Qiskit displays highest-index bit on the left.
        chars = []
        for char_index in range(n_chars):
            chunk = bits[char_index * 8:(char_index + 1) * 8]
            value = sum(int(bit) << bit_index for bit_index, bit in enumerate(chunk))
            chars.append(chr(value))
        text = "".join(chars)
        decoded[text] = decoded.get(text, 0) + count
    return decoded

measured = cord_circuit.copy()
measured.measure_all()
counts = simulator.run(measured, shots=100).result().get_counts()
print(decode_counts(counts, n_chars=len(cord_text)))

## 7. Cord notation summary

```text
Cord("HI")
  ├── Knot("H"): 8 qubits
  └── Knot("I"): 8 qubits
Total: 16 qubits
```

- A **knot** groups qubits and labels what they encode.
- A **cord** orders knots into a string.
- Encoding prepares a computational-basis state.
- A line between knots means sequence/composition only. It does not claim entanglement.

### Check your understanding
1. How many qubits are in `QCord("HELLO")` under this encoding?
2. Does putting characters in the same cord automatically entangle them?
3. What does $X$ do to $|0\rangle$?
4. Why is this a representation/teaching tool rather than a faster text processor?

### Next notebook
**Notebook 02: Transforming knots with gates** — apply $X$ to a selected bit, derive the changed state with LaTeX, and see which character is measured. Later: Hadamard superposition, rotations, and entanglement.